In [0]:
%run ../init_framework

In [0]:
# Schema Declared
LKP_SCHEMA_DDL = "loan_purpose STRING, is_active BOOLEAN"

# Read lookup file
df_lkp_raw = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(LKP_SCHEMA_DDL)
    .load(f"{SRC_REFERENCE}loan_purpose_lookup.csv")
)

# Create Temp View
df_lkp_raw.createOrReplaceTempView("tmp_loan_purpose_seed")

# Execute MERGE
spark.sql(
    f"""
    MERGE INTO {REF_LOAN_PURPOSES} AS target
    USING tmp_loan_purpose_seed AS source
    ON target.loan_purpose = source.loan_purpose
    WHEN MATCHED THEN
      UPDATE SET target.is_active = source.is_active
    WHEN NOT MATCHED THEN
      INSERT (loan_purpose, is_active) VALUES (source.loan_purpose, source.is_active)
"""
)